Problem Statement: Spreadsheet Simulator

You are tasked with designing a basic spreadsheet simulator. The simulator should allow the assignment of values or formulas to cells in a grid-like spreadsheet, and the retrieval of those values or computed results. The simulator must support two key functionalities:

    Assign a value or formula to a cell:
        Cells are identified by labels like A1, B2, etc., where the letter refers to the column and the number refers to the row.
        You can assign:
            A constant value (e.g., 5 or "Hello") to a cell.
            A formula (e.g., =A1+B2, =SUM(A1:A3)).
        Formulas should support basic arithmetic operations (+, -, *, /) and aggregation functions like SUM over ranges.

    Get the value of a cell:
        If a cell contains a constant, return that value directly.
        If the cell contains a formula, evaluate it and return the result.
        If the formula references other cells, recursively calculate the referenced cells' values.

Requirements:

    The simulator should handle cyclic references and raise an error if a cycle is detected.
    The system should cache computed values to avoid recalculating the same cell multiple times, updating only when dependencies change.
    Implement basic parsing to distinguish between values and formulas.

In [5]:
import re

class Spreadsheet:
    def __init__(self):
        self.cells = {}  # Stores the value or formula for each cell
        self.cache = {}  # Caches the computed value of each cell
    
    def set(self, cell, value_or_formula):
        """Sets a value or formula in a cell."""
        self.cells[cell] = value_or_formula
        if cell in self.cache:
            del self.cache[cell]  # Clear cached value when cell is updated
    
    def get(self, cell, visited=None):
        """Gets the value of a cell, evaluating formulas if necessary."""
        if visited is None:
            visited = set()
        if cell in visited:
            raise ValueError(f"Cyclic reference detected in cell {cell}")
        
        if cell in self.cache:
            return self.cache[cell]
        
        value = self.cells.get(cell)
        if isinstance(value, (int, float)):  # If it's a constant
            return value
        elif isinstance(value, str):
            if value.startswith('='):
                visited.add(cell)
                result = self._evaluate_formula(value[1:], visited)
                visited.remove(cell)
                self.cache[cell] = result
                return result
            else:
                return value
        else:
            raise ValueError(f"Cell {cell} is empty or contains an invalid value.")
    
    def _evaluate_formula(self, formula, visited):
        """Evaluates a formula."""
        # Find all cell references (e.g., A1, B2) in the formula
        tokens = re.split(r'([A-Z]+\d+)', formula)
        
        # Replace cell references with their evaluated values
        evaluated_tokens = []
        for token in tokens:
            if re.match(r'[A-Z]+\d+', token):  # If it's a cell reference
                evaluated_tokens.append(str(self.get(token, visited)))
            else:
                evaluated_tokens.append(token)
        
        # Join the tokens back into a valid Python expression and evaluate it
        expression = ''.join(evaluated_tokens)
        try:
            result = eval(expression)  # This evaluates the arithmetic expression
            return result
        except Exception as e:
            raise ValueError(f"Error evaluating formula '{formula}': {e}")

# Example usage:
spreadsheet = Spreadsheet()
spreadsheet.set('A1', 10)
spreadsheet.set('B1', 5)
spreadsheet.set('C1', '=A1+B1')
print(spreadsheet.get('C1'))  # Output: 15

spreadsheet.set('A2', 3)
spreadsheet.set('A3', 7)
spreadsheet.set('B2', '=A1+A2+A3')
print(spreadsheet.get('B2'))  # Output: 20


15
20
